# 서울시 부동산 2024 데이터 해설/정답


In [1]:
import pandas as pd

# 서울시 부동산 데이터는 보통 cp949 인코딩이 많습니다.
df = pd.read_csv("2024.csv", encoding="cp949", low_memory=False)
print(df.shape)
df.head()

(77523, 21)


,접수연도,자치구코드,자치구명,법정동코드,법정동명,지번구분,지번구분명,본번,부번,건물명,...,물건금액(만원),건물면적(㎡),토지면적(㎡),층,권리구분,취소일,건축년도,건물용도,신고구분,신고한 개업공인중개사 시군구명
0,2024,11380,은평구,10200,녹번동,1.0,대지,0086,0125,성원빌라(86-125),...,16400,53.07,25.0000,4.0,NaN,NaN,1991.0,연립다세대,중개거래,서울 은평구
1,2024,11500,강서구,10300,화곡동,1.0,대지,0924,0004,팔팔빌라,...,35000,39.24,27.0000,2.0,NaN,NaN,1986.0,연립다세대,직거래,NaN
2,2024,11560,영등포구,11100,당산동1가,1.0,대지,0143,0000,다솔시티하임,...,30000,29.54,14.0000,5.0,NaN,NaN,2017.0,연립다세대,직거래,NaN
3,2024,11560,영등포구,12800,양평동4가,1.0,대지,0220,0000,해울빌,...,36000,40.41,63.6400,3.0,NaN,NaN,2021.0,오피스텔,중개거래,서울 용산구
4,2024,11410,서대문구,11200,대현동,1.0,대지,0090,0077,유씨유 이대,...,19300,17.31,30.8725,5.0,NaN,NaN,2018.0,오피스텔,중개거래,서울 서대문구


### 문제 1 해설 — 시간: `계약일` -> datetime 변환

서울시 데이터는 날짜가 `YYYYMMDD` 정수로 오는 경우가 많습니다.

- `format`을 지정하면 파싱이 **빠르고** 일관적
- 이상한 값이 섞여 있어도 `errors='coerce'`면 안전하게 `NaT`로 처리됩니다.

In [2]:
df["계약일_dt"] = pd.to_datetime(
    df["계약일"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

df["계약일_dt"].dtype


dtype('<M8[ns]')

### 문제 2 해설 — 시간: 계약 연/월/일 파생컬럼 만들기

`datetime`으로 바꿔두면 `.dt`로 필요한 조각을 쉽게 뽑을 수 있습니다.

- 월별/분기별 분석할 때 이 파생컬럼이 거의 필수

In [3]:
df["계약연"] = df["계약일_dt"].dt.year
df["계약월"] = df["계약일_dt"].dt.month
df["계약일자"] = df["계약일_dt"].dt.day

df[["계약일", "계약일_dt", "계약연", "계약월", "계약일자"]].head()


,계약일,계약일_dt,계약연,계약월,계약일자
0,20241014,2024-10-14,2024,10,14
1,20241014,2024-10-14,2024,10,14
2,20241014,2024-10-14,2024,10,14
3,20241014,2024-10-14,2024,10,14
4,20241014,2024-10-14,2024,10,14


### 문제 3 해설 — 그룹화+agg: 자치구별 거래건수 & 평균 거래금액

`agg`로 여러 통계를 한 번에 뽑는 게 핵심.

- `size`는 행 개수(결측 영향 적음)
- 평균은 금액 컬럼 기준

In [4]:
gu_summary = (
    df.groupby("자치구명")
      .agg(
          count=("물건금액(만원)", "size"),
          price_mean=("물건금액(만원)", "mean"),
      )
      .sort_values("count", ascending=False)
)
gu_summary.head(10)


,count,price_mean
자치구명,,
송파구,5111,124270.190765
강서구,4936,51481.504254
강동구,4289,86975.403357
강남구,4030,201799.790571
마포구,4028,92563.339126
은평구,3913,51542.257858
노원구,3855,57707.193515
영등포구,3488,92959.351778
양천구,3474,83843.756189


### 문제 4 해설 — 복수 기준 그룹화: (자치구, 건물용도) 요약

복수 기준 groupby는 나중에 `unstack`/`pivot_table`로 예쁘게 펼치기 좋습니다.

- 중앙값까지 같이 보면 이상치 영향도 같이 체크 가능

In [5]:
gu_type = (
    df.groupby(["자치구명", "건물용도"])
      .agg(
          count=("물건금액(만원)", "size"),
          price_mean=("물건금액(만원)", "mean"),
          price_median=("물건금액(만원)", "median"),
      )
      .sort_values("count", ascending=False)
)
gu_type.head(10)


,,count,price_mean,price_median
자치구명,건물용도,,,
노원구,아파트,3408,61722.099178,57750.0
송파구,아파트,3406,163256.292719,154350.0
강동구,아파트,2906,110947.762560,112400.0
강남구,아파트,2878,243734.100417,220000.0
성북구,아파트,2512,79731.789411,78000.0
성동구,아파트,2410,140652.752697,130000.0
영등포구,아파트,2356,116823.932513,103500.0
강서구,아파트,2314,81629.970182,77000.0
마포구,아파트,2234,129292.316025,127000.0


### 문제 5 해설 — transform: 자치구 평균 금액 대비(차이) 컬럼 만들기

집계표로 평균을 구하면 행 수가 줄어들어서 원본에 붙이기 번거롭죠.

- `transform('mean')`이면 원본 길이를 유지한 채로 평균이 각 행에 복제됩니다.

In [6]:
df["구평균금액"] = df.groupby("자치구명")["물건금액(만원)"].transform("mean")
df["금액_구평균차"] = df["물건금액(만원)"] - df["구평균금액"]

df[["자치구명", "물건금액(만원)", "구평균금액", "금액_구평균차"]].head()


,자치구명,물건금액(만원),구평균금액,금액_구평균차
0,은평구,16400,51542.257858,-35142.257858
1,강서구,35000,51481.504254,-16481.504254
2,영등포구,30000,92959.351778,-62959.351778
3,영등포구,36000,92959.351778,-56959.351778
4,서대문구,19300,73761.722085,-54461.722085


### 문제 6 해설 — unstack: 자치구 × 건물용도 거래건수 표

MultiIndex Series -> `unstack()` 하면 바로 리포트 표가 됩니다.

- 행: 자치구
- 열: 건물용도
- 값: 거래건수

In [7]:
count_table = (
    df.groupby(["자치구명", "건물용도"])
      .size()
      .unstack(fill_value=0)
)
count_table.head()


건물용도,단독다가구,아파트,연립다세대,오피스텔
자치구명,,,,
강남구,76,2878,466,610
강동구,69,2906,980,334
강북구,138,771,1089,88
강서구,56,2314,1933,633
관악구,142,1221,978,299


### 문제 7 해설 — 단가 계산: ㎡당 단가(만원/㎡) + 자치구별 평균 단가

apply(axis=1)도 가능하지만, 이런 계산은 **벡터 연산이 더 빠르고 깔끔**합니다.

- 면적이 결측이면 단가도 NaN으로 자연스럽게 전파됩니다.

In [8]:
df["단가_만원_㎡"] = df["물건금액(만원)"] / df["건물면적(㎡)"]

unit_by_gu = (
    df.groupby("자치구명")["단가_만원_㎡"]
      .mean()
      .sort_values(ascending=False)
      .head(10)
)
unit_by_gu


자치구명
강남구     2318.698983
서초구     2301.577800
용산구     1990.652086
성동구     1750.778597
송파구     1653.118879
마포구     1419.247629
영등포구    1303.586020
강동구     1303.513714
동작구     1301.639647
중구      1283.097138
Name: 단가_만원_㎡, dtype: float64

### 문제 8 해설 — pivot_table: 계약월 × 건물용도 평균 단가 표

`pivot_table`은 중복 조합이 있어도 `aggfunc`로 알아서 집계해줍니다.

- 월×용도 매트릭스는 트렌드 보기 좋음!

In [9]:
unit_pivot = df.pivot_table(
    values="단가_만원_㎡",
    index="계약월",
    columns="건물용도",
    aggfunc="mean",
    fill_value=0
).sort_index()

unit_pivot


건물용도,단독다가구,아파트,연립다세대,오피스텔
계약월,,,,
1,1007.412476,1393.010042,829.675603,804.151542
2,980.511842,1408.765115,810.075095,924.802340
3,929.984162,1454.950332,811.339499,905.800240
4,962.177268,1489.978889,804.235511,834.455623
5,905.263346,1541.045312,830.641028,849.390791
6,1051.854588,1616.119176,864.443913,839.653539
7,1115.365292,1564.964064,917.520403,835.846893
8,1100.155707,1530.733079,921.635659,875.436708
9,856.499656,1466.881026,862.426504,842.215856


### 문제 9 해설 — melt: 8번 피벗 표를 long으로

시각화/분석용 데이터는 보통 long(tidy)이 편합니다.

- `reset_index()` -> `melt()` 순서 기억!

In [10]:
unit_long = (
    unit_pivot.reset_index()
              .melt(id_vars="계약월", var_name="건물용도", value_name="avg_unit_price")
)
unit_long.head()


,계약월,건물용도,avg_unit_price
0,1,단독다가구,1007.412476
1,2,단독다가구,980.511842
2,3,단독다가구,929.984162
3,4,단독다가구,962.177268
4,5,단독다가구,905.263346


### 문제 10 해설 — pivot: 9번 long을 다시 wide로 복원

`pivot`은 (index, columns) 조합이 유일할 때만 깔끔하게 됩니다.

- 지금은 우리가 피벗에서 만든 long이라 조합이 유일 -> 복원 OK

In [11]:
unit_wide_back = unit_long.pivot(
    index="계약월",
    columns="건물용도",
    values="avg_unit_price"
).sort_index()

print(unit_wide_back.equals(unit_pivot))
unit_wide_back


True


건물용도,단독다가구,아파트,연립다세대,오피스텔
계약월,,,,
1,1007.412476,1393.010042,829.675603,804.151542
2,980.511842,1408.765115,810.075095,924.802340
3,929.984162,1454.950332,811.339499,905.800240
4,962.177268,1489.978889,804.235511,834.455623
5,905.263346,1541.045312,830.641028,849.390791
6,1051.854588,1616.119176,864.443913,839.653539
7,1115.365292,1564.964064,917.520403,835.846893
8,1100.155707,1530.733079,921.635659,875.436708
9,856.499656,1466.881026,862.426504,842.215856


### 문제 11 해설 — 문자열 정리: `건물명` 클린 컬럼 만들기(괄호 제거 포함)

`str.replace`에서 정규표현식(`regex=True`)을 쓰면 괄호 같은 문자 처리도 깔끔합니다.

- 결측을 먼저 채우지 않으면 `.str` 작업이 꼬일 수 있어요.

In [12]:
df["건물명_clean"] = (
    df["건물명"]
      .fillna("미상")
      .astype(str)
      .str.replace("(", "")
      .str.replace(")", "")
      .str.strip()
)

df[["건물명", "건물명_clean"]].head()


,건물명,건물명_clean
0,성원빌라(86-125),성원빌라86-125
1,팔팔빌라,팔팔빌라
2,다솔시티하임,다솔시티하임
3,해울빌,해울빌
4,유씨유 이대,유씨유 이대


### 문제 12 해설 — 문자열 contains: 빌라 거래 건수 + 자치구 TOP 5

`contains`는 텍스트 필터링의 기본.

- boolean mask를 만들고 `sum()`하면 True 개수(=건수)입니다.

In [ ]:
# na=False : 문자열을 검사하다가 결측치(NaN/NA)를 만나면, 그 결과를 False로 채우라는 뜻
villa_mask = df["건물명_clean"].str.contains("빌라", na=False)

villa_count = villa_mask.sum()
print("빌라 거래 건수:", villa_count)

villa_top5 = (
    df.loc[villa_mask]
      .groupby("자치구명")
      .size()
      .sort_values(ascending=False)
      .head(5)
)
villa_top5


빌라 거래 건수: 4021


자치구명
강서구    346
은평구    284
강북구    276
송파구    259
관악구    253
dtype: int64

### 문제 13 해설 — 문자열: 법정동명 마지막 글자 빈도 TOP 10

가끔은 마지막 글자(동/가/리 등)만 봐도 데이터 품질 체크가 됩니다.

- 결측이 있을 수 있으니 `astype(str)`로 안전하게

In [15]:
df["동_끝글자"] = df["법정동명"].astype(str).str[-1]
df["동_끝글자"].value_counts().head(10)

동_끝글자
동    73522
가     4001
Name: count, dtype: int64

### 문제 14 해설 — map: 신고구분 설명 라벨 붙이기

라벨링은 `map`이 제일 편합니다.

- 매핑에 없는 값은 NaN이 되므로 `fillna('기타')`로 마무리

In [17]:
report_map = {
    "중개거래": "중개사 통해 거래",
    "직거래": "직접 거래",
}

df["신고구분_설명"] = df["신고구분"].map(report_map).fillna("기타")
df["신고구분_설명"].value_counts()


신고구분_설명
중개사 통해 거래    68679
직접 거래         8842
기타               2
Name: count, dtype: int64

### 문제 15 해설 — 시간: `취소일` datetime 변환 + 계약일과의 차이(일수)

취소일은 결측이 많습니다. 그래서 `errors='coerce'`가 거의 필수입니다.

In [22]:
df["취소일_dt"] = pd.to_datetime(
    df["취소일"],
    format="%Y%m%d",
    errors="coerce"
)

df["days_to_cancel"] = (df["취소일_dt"] - df["계약일_dt"]).dt.days

print("days_to_cancel 결측치:", df["days_to_cancel"].isna().sum())
df[["계약일_dt", "취소일_dt", "days_to_cancel"]].head()


days_to_cancel 결측치: 74351


,계약일_dt,취소일_dt,days_to_cancel
0,2024-10-14,NaT,NaN
1,2024-10-14,NaT,NaN
2,2024-10-14,NaT,NaN
3,2024-10-14,NaT,NaN
4,2024-10-14,NaT,NaN


### 문제 16 해설 — merge: 자치구 -> 권역 매핑 붙이기 + 권역별 평균금액

권역처럼 '외부 기준표'로 붙여야 하는 정보는 merge가 정석입니다.

- `left merge`로 원본 행 유지
- 권역이 NaN이면 매핑 누락이 있는지 체크 포인트

In [23]:
region_map = {
    # 도심권
    "종로구": "도심권", "중구": "도심권", "용산구": "도심권",
    # 동북권
    "성동구": "동북권", "광진구": "동북권", "동대문구": "동북권", "중랑구": "동북권",
    "성북구": "동북권", "강북구": "동북권", "도봉구": "동북권", "노원구": "동북권",
    # 서북권
    "은평구": "서북권", "서대문구": "서북권", "마포구": "서북권",
    # 서남권
    "양천구": "서남권", "강서구": "서남권", "구로구": "서남권", "금천구": "서남권",
    "영등포구": "서남권", "동작구": "서남권", "관악구": "서남권",
    # 동남권
    "서초구": "동남권", "강남구": "동남권", "송파구": "동남권", "강동구": "동남권",
}

region_df = pd.DataFrame({
    "자치구명": list(region_map.keys()),
    "권역": list(region_map.values()),
})

df2 = df.merge(region_df, on="자치구명", how="left")

region_summary = (
    df2.groupby("권역")["물건금액(만원)"]
       .mean()
       .sort_values(ascending=False)
)
region_summary


권역
동남권    145584.275667
도심권    118929.196002
서북권     72721.661781
동북권     69686.750280
서남권     68439.923662
Name: 물건금액(만원), dtype: float64

### 문제 17 해설 — set_index: 계약일을 인덱스로 두고 특정 월 데이터 슬라이싱

DatetimeIndex를 만들면 문자열 슬라이싱이 꽤 강력합니다.

- 먼저 `sort_index()`는 거의 필수(정렬돼 있어야 슬라이싱이 안정적)
- 분석 후 `reset_index()`로 원복

In [24]:
df_idx = df.set_index("계약일_dt").sort_index()

oct_2024 = df_idx.loc["2024-10"]
print("2024-10 거래 건수:", len(oct_2024))

df_back = df_idx.reset_index()
df_back.head()


2024-10 거래 건수: 897


,계약일_dt,접수연도,자치구코드,자치구명,법정동코드,법정동명,지번구분,지번구분명,본번,부번,...,계약월,계약일자,구평균금액,금액_구평균차,단가_만원_㎡,건물명_clean,동_끝글자,신고구분_설명,취소일_dt,days_to_cancel
0,2021-02-08,2024,11305,강북구,10100,미아동,NaN,NaN,NaN,NaN,...,2,8,41186.525887,122813.474113,559.574178,미상,동,기타,2024-09-05,1305.0
1,2021-07-19,2024,11440,마포구,11000,노고산동,NaN,NaN,NaN,NaN,...,7,19,92563.339126,102436.660874,1238.488409,미상,동,기타,2024-04-17,1003.0
2,2022-07-08,2024,11530,구로구,10700,개봉동,1.0,대지,0170,0033,...,7,8,52665.414107,-43475.414107,171.423242,금석연립170-33,동,중개사 통해 거래,2024-02-07,579.0
3,2022-07-08,2024,11530,구로구,10700,개봉동,1.0,대지,0170,0033,...,7,8,52665.414107,-43475.414107,171.423242,금석연립170-33,동,중개사 통해 거래,2024-02-07,579.0
4,2022-07-08,2024,11530,구로구,10700,개봉동,1.0,대지,0170,0033,...,7,8,52665.414107,-42925.414107,181.682522,금석연립170-33,동,중개사 통해 거래,2024-02-07,579.0


### 문제 18 해설 — concat: 신고구분별 월별 평균금액 요약표 합치기

조건별로 나눠서 만든 요약표를 하나로 합칠 때 `concat`이 딱입니다.

- `ignore_index=True`로 인덱스 정리
- `assign(신고구분=...)`으로 출처 라벨 붙이기

In [ ]:
broker = df[df["신고구분"] == "중개거래"]
direct = df[df["신고구분"] == "직거래"]

broker_month = (
    broker.groupby("계약월")["물건금액(만원)"]
          .mean()
          .reset_index()
          .assign(신고구분="중개거래")
)

direct_month = (
    direct.groupby("계약월")["물건금액(만원)"]
          .mean()
          .reset_index()
          .assign(신고구분="직거래")
)

monthly_compare = pd.concat([broker_month, direct_month], axis=0, ignore_index=True)
monthly_compare.sort_values(["계약월", "신고구분"], ignore_index=True).head(10)

,계약월,물건금액(만원),신고구분
0,1,79033.991872,중개거래
1,1,46307.354217,직거래
2,2,82230.325858,중개거래
3,2,45921.910732,직거래
4,3,85430.130231,중개거래
5,3,46617.140444,직거래
6,4,88740.923516,중개거래
7,4,52929.422454,직거래
8,5,94630.434091,중개거래
9,5,45177.020513,직거래
